# Preprocess Collected Data
* Preprocess the collected data
* Label the collected data
* Combine the data with other public labelled datasets to ensure robustness

In [ ]:
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
from transformers import pipeline
import torch
import malaya
from collections import Counter
from datasets import load_dataset

### Data Ingestion and Inspection

In [ ]:
df = pd.read_csv("../data/malaysia_food_comments.csv")
df.head()

In [ ]:
# Data Inspection
print(df.shape)
print()
print(df.dtypes)
print()
print(df.isnull().sum())

### Data Preprocessing

In [ ]:
# Data Preprocessing
def preprocess_comment(text):
    if not isinstance(text, str):
        return ""
    
    # Strip HTML tags
    text = BeautifulSoup(text, 'html.parser').get_text()
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Replace mentions and hashtags
    text = re.sub(r'@\w+', '@user', text)
    text = re.sub(r'#\w+', '', text)
    
    # Remove HTML entities
    text = re.sub(r'&\w+;', '', text)
    
    # Normalise repeated characters (e.g. "sooooo good" → "so good")
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['comment_clean'] = df['comment'].apply(preprocess_comment)

In [ ]:
# Filter Noise
def is_garbage(text):
    if not isinstance(text, str):
        return True
    # Mostly numbers/symbols (no real word characters)
    if re.sub(r'[\d\s\W]', '', text) == '':
        return True
    # Too few actual characters after stripping non-alphanumeric
    if len(re.sub(r'[^\w]', '', text)) < 3:
        return True
    # Repeated single word (e.g. "haha haha haha")
    words = text.split()
    if len(words) >= 3 and len(set(words)) == 1:
        return True
    # Spam-like: excessive punctuation/caps ratio
    if len(text) > 0 and sum(1 for c in text if c in '!?') / len(text) > 0.3:
        return True
    return False

# Drop nulls and empty strings
df = df[df['comment_clean'].str.len() > 0].copy()

# Drop duplicates
df = df.drop_duplicates(subset='comment_clean').reset_index(drop=True)

# Drop garbage
df = df[~df['comment_clean'].apply(is_garbage)].reset_index(drop=True)

# Drop very short comments
df = df[df['comment_clean'].str.split().str.len() >= 3].reset_index(drop=True)

# Drop very long comments (outliers that may break tokenizer max_length)
df = df[df['comment_clean'].str.split().str.len() <= 200].reset_index(drop=True)

print(f"Clean dataset size: {len(df)}")

In [ ]:
df[['comment', 'comment_clean']].sample(10)

### Label Data

In [ ]:
device = 0 if torch.cuda.is_available() else -1
print(f"Using: {'GPU' if device == 0 else 'CPU'}")

In [ ]:
# Model 1: Twitter-RoBERTa
roberta_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=device,
    truncation=True,
    max_length=512
)

# Model 2: XLM-RoBERTa
xlmr_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    device=device,
    truncation=True,
    max_length=512
)

# Model 3: Multilingual 
multi_pipe = pipeline(
    "sentiment-analysis",
    model="tabularisai/multilingual-sentiment-analysis",
    tokenizer="tabularisai/multilingual-sentiment-analysis",
    device=device,
    truncation=True,
    max_length=512
)

# # Model 4: Malaya (got issues)
# malaya_model = malaya.sentiment.huggingface(
#     model='mesolitica/sentiment-analysis-nanot5-small-malaysian-cased'
# )

In [ ]:
from tqdm import tqdm

BATCH_SIZE = 32

def run_pipeline_batched(pipe, texts):
    results = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch = texts[i:i+BATCH_SIZE]
        batch_results = pipe(batch, truncation=True, max_length=512)
        results.extend(batch_results)
    return results

# def run_malaya_batched(model, texts):
#     results = []
#     for i in tqdm(range(0, len(texts), BATCH_SIZE)):
#         batch = texts[i:i+BATCH_SIZE]
#         probs = model.predict_proba(batch)
#         for prob in probs:
#             label = max(prob, key=prob.get)
#             score = prob[label]
#             results.append({'label': label, 'score': score})
#     return results

In [ ]:
texts = df['comment_clean'].tolist()

print("Running Twitter-RoBERTa...")
roberta_results = run_pipeline_batched(roberta_pipe, texts)

print("Running XLM-RoBERTa...")
xlmr_results = run_pipeline_batched(xlmr_pipe, texts)

print("Running Multilingual...")
multi_results = run_pipeline_batched(multi_pipe, texts)

# print("Running Malaya...")
# malaya_results = run_malaya_batched(malaya_model, texts)

In [ ]:
def normalise_label(label: str) -> str:
    label = label.lower().strip()
    
    if label == 'very positive':
        return 'positive'
    if label == 'very negative':
        return 'negative'
    
    return label

df['roberta_label'] = [normalise_label(r['label']) for r in roberta_results]
df['roberta_score'] = [r['score'] for r in roberta_results]

df['xlmr_label'] = [normalise_label(r['label']) for r in xlmr_results]
df['xlmr_score'] = [r['score'] for r in xlmr_results]

df['multi_label'] = [normalise_label(r['label']) for r in multi_results]
df['multi_score'] = [r['score'] for r in multi_results]

# df['malaya_label'] = [normalise_label(r['label']) for r in malaya_results]
# df['malaya_score'] = [r['score'] for r in malaya_results]

In [ ]:
# Voting
def vote(row):
    labels = [row['roberta_label'], row['xlmr_label'], row['multi_label']]
    scores = {
        row['roberta_label']: row['roberta_score'],
        row['xlmr_label']:    row['xlmr_score'],
        row['multi_label']:  row['multi_score'],
    }
    
    count = Counter(labels)
    most_common_label, most_common_count = count.most_common(1)[0]
    
    if most_common_count == 3:
        agreement = 'full'
        confidence = np.mean([row['roberta_score'], row['xlmr_score'], row['multi_score']])
    elif most_common_count == 2:
        agreement = 'majority'
        # Average confidence of the 2 agreeing models only
        agreeing_scores = [
            score for label, score in zip(
                [row['roberta_label'], row['xlmr_label'], row['multi_label']],
                [row['roberta_score'], row['xlmr_score'], row['multi_score']]
            ) if label == most_common_label
        ]
        confidence = np.mean(agreeing_scores)
    else:
        agreement = 'none'
        most_common_label = 'review'
        confidence = 0.0
    
    return pd.Series({
        'final_label': most_common_label,
        'confidence':  round(confidence, 4),
        'agreement':   agreement
    })

df[['final_label', 'confidence', 'agreement']] = df.apply(vote, axis=1)

In [ ]:
print(df['agreement'].value_counts())
print(df['final_label'].value_counts())

# # Flag low confidence majority cases for manual review (gave up manual review)
# manual_review = df[
#     (df['agreement'] == 'none') 
#     | ((df['agreement'] == 'majority') & (df['confidence'] < 0.6))
# ]
# print(f"\nSamples needing manual review: {len(manual_review)}")
# manual_review[['comment_clean', 'roberta_label', 'xlmr_label', 'multi_label', 'final_label', 'confidence']].sample(10)

In [ ]:
high_conf = df[
    (df['agreement'] == 'full') |
    ((df['agreement'] == 'majority') & (df['confidence'] >= 0.6))
].copy()

high_conf = high_conf[['comment_clean', 'final_label']].rename(columns={'comment_clean': 'text', 'final_label': 'label'})

# Standardise labels
label_map_str = {'positive': 2, 'neutral': 1, 'negative': 0}
high_conf['label'] = high_conf['label'].map(label_map_str)

high_conf.head()

In [ ]:
high_conf.to_csv('../data/labeled_comments.csv', index=False, encoding='utf-8-sig')
print(f"Saved {len(high_conf)} labeled samples")

### Balance Training Dataset

In [ ]:
# Ingest Cleaned Data
df_labeled = pd.read_csv("../data/labeled_comments.csv")
df_labeled['source'] = 'youtube_scraped'
df_labeled.head()


In [ ]:
print(df_labeled['label'].value_counts())
print()
print(df_labeled['label'].value_counts(normalize=True).mul(100).round(1))

In [ ]:
# Ingest public dataset
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")
df_public = pd.DataFrame(dataset['train'])
print(df_public['label'].value_counts())
df_public.head()

In [ ]:
# Add source label
df_public['source'] = 'tweet_eval'
df_public.head()

In [ ]:
# Number of rows needed to balance the data
scraped_counts = df_labeled['label'].value_counts()
print("Scraped counts:\n", scraped_counts)

needed = {
    0: scraped_counts[2] - scraped_counts[0],  # negative needed
    1: scraped_counts[2] - scraped_counts[1],  # neutral needed
    2: 0                                       # positive already dominant
}

needed

In [ ]:
# Sample needed rows per class from public data
public_samples = []
for label, n in needed.items():
    if n > 0:
        pool = df_public[df_public['label'] == label]
        sampled = pool.sample(min(n, len(pool)), random_state=42)
        public_samples.append(sampled)

df_public_sampled = pd.concat(public_samples, ignore_index=True)
print("Public sampled distribution:\n", df_public_sampled['label'].value_counts())

In [ ]:
# Combine
df_combined = pd.concat([df_labeled, df_public_sampled], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print("Combined distribution:")
print(df_combined['label'].value_counts())
print()
print(df_combined['source'].value_counts())

print(f"\nTotal: {len(df_combined)}")

df_combined.sample(10)

In [ ]:
df_combined.to_csv('../data/labeled_scraped_and_public_data.csv', index=False, encoding='utf-8-sig')
print("Saved labeled_scraped_and_public_data.csv")